In [11]:
%pip install sqlalchemy fairscape_models

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [102]:
from fairscape_models.rocrate import (
	ROCrateV1_2,
	ROCrateMetadataElem, 
	ROCrateMetadataFileElem,
)
from fairscape_models.dataset import Dataset
from fairscape_models.software import Software
from fairscape_models.computation import Computation
from fairscape_models.fairscape_base import IdentifierValue
from fairscape_models.biochem_entity import BioChemEntity
from fairscape_models.schema import Schema, Property

import json
import pathlib
import zipfile

import sqlalchemy as sa
from sqlalchemy.orm import declarative_base, Mapped, mapped_column, relationship
import enum
from pydantic import BaseModel, ConfigDict, Field
from typing import List, Optional,Union

In [13]:
# Other List Tables

class MetadataTypeSQL():
	pass

class D4DCorrectionsSQL():
	pass

class FunderSQL():
	pass

class CitationSQL():
	pass

class AssociatedPublicationsSQL():
	pass

class AdditionalPropertySQL():
	pass

class EthicalReviewSQL():
	pass

class IRBSQL():
	pass

class AboutSQL():
	pass

class RAIDataLimitationsSQL():
	pass

class RAIDataBiasesSQL():
	pass

class RAIDataUseCasesSQL():
	pass

class RAIDataReleaseMaintenancePlanSQL():
	pass

class RAIDataCollectionPlanSQL():
	pass

class RAIDataCollectionSQL():
	pass


class RAIDataCollectionTypeSQL():
	pass

class RAIDataCollectionMissingDataSQL():
	pass

class RAIDataCollectionRawDataSQL():
	pass

class RAIDataCollectionTimeframeSQL():
	pass

class RAIDataImputationProtocolSQL():
	pass

class RAIDataManipulationProtocolSQL():
	pass

class RAIDataPreprocessingProtocolSQL():
	pass

class RAIDataAnnotationProtocolSQL():
	pass

class RAIDataAnnotationPlatformSQL():
	pass

class RAIDataAnnotationAnalysisSQL():
	pass

class RAIPersonalSensitiveInformationSQL():
	pass

class RAIDataSocialImpactSQL():
	pass

class RAIAnnotationsPerItemSQL():
	pass

class RAIMachineAnnotationToolsSQL():
	pass

class CompletenessSQL():
	pass

class ProhibitedUsesSQL():
	pass

### Set Up Tables

In [14]:
Base = declarative_base()

class MetadataTypeEnumSQL(enum.Enum):
	ROCRATE = "ROCRATE"
	SOFTWARE = "SOFTWARE"
	DATASET = "DATASET"
	COMPUTATION = "COMPUTATION"
	ANNOTATION = "ANNOTATION"
	EXPERIMENT = "EXPERIMENT"
	PATIENT = "PATIENT"
	CREATIVE_WORK = "CREATIVE_WORK"
	SAMPLE = "SAMPLE"
	SCHEMA = "SCHEMA"
	BIO_CHEM_ENTITY = "BIO_CHEM_ENTITY"
	MEDICAL_CONDITION = "MEDICAL_CONDITION"
	PERSON = "PERSON"
	ORGANIZATION = "ORGANIZATION"
	DEFINED_TERM = "DEFINED_TERM"
	NONE = "NONE"

class IdentifiersSQL(Base):
	__tablename__ = 'identifier'
	__table_args__ = {"extend_existing": True}  
	guid: str = sa.Column('guid', sa.String, primary_key=True)
	name: str = sa.Column('name', sa.String)
	metadataType: Mapped[MetadataTypeEnumSQL] = mapped_column(sa.Enum(MetadataTypeEnumSQL))

class KeywordSQL(Base):
	__tablename__ = 'keyword_table'
	__table_args__ = {"extend_existing": True}
	id: Mapped[int] = mapped_column(primary_key=True)
	guid: Mapped[str]
	keywordValue: Mapped[str] 
	
class AuthorSQL(Base):
	__tablename__ = 'author'
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	name: Mapped[str]
	orcid: Mapped[Optional[str]] = mapped_column(default=None)

class AuthorIdentifierSQL(Base):
	__tablename__ = 'author_identifier'
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	author_id: Mapped[int]
	identifier_guid: Mapped[str]


class MembershipSQL(Base):
	__tablename__ = 'membership'
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	parentGUID: Mapped[str] 
	parentType: Mapped[MetadataTypeEnumSQL] = mapped_column(sa.Enum(MetadataTypeEnumSQL))
	childGUID: Mapped[str] 
	childType: Mapped[MetadataTypeEnumSQL] = mapped_column(sa.Enum(MetadataTypeEnumSQL))


class ComputationUsedDatasetSQL():
	__tablename__ = "used_dataset"
	__table_args__ = {'extend_existing': True}  
	computationGUID: Mapped[str]
	datasetGUID: Mapped[str]


class ComputationGeneratedDatasetSQL():
	__tablename__ = "generated_dataset"
	__table_args__ = {'extend_existing': True}  
	computationGUID: Mapped[str]
	datasetGUID: Mapped[str]


class EntitySQL():
	__table_args__ = {"extend_existing": True}  
	id: Mapped[int] = mapped_column(primary_key=True)
	guid: Mapped[str] = sa.Column('guid', sa.String)
	name: Mapped[str] = sa.Column('name', sa.String)
	description: Mapped[str] = sa.Column('description', sa.String)
	datePublished: Mapped[Optional[str]] 


class HasContent():
	contentURL: Mapped[Optional[str]] = sa.Column('contentURL', sa.String, default=None)

class Versioned():
	version: Mapped[str]

class ROCrateMetadataElemSQL(EntitySQL, HasContent, Versioned, Base):
	__tablename__ = 'rocrate'
	__table_args__ = {"extend_existing": True}  
	license: Mapped[str]

	# is part of add to membership table 
	#isPartOf: Mapped[Optional[List["MembershipSQL"]]] = relationship(back_populates="parentGUID")
	#datePublished: datetime
	#about: str
	#publisher: str


class DatasetSQL(EntitySQL, HasContent, Versioned, Base):
	__tablename__ = 'dataset'
	fileFormat: Mapped[str]
	generatedBy: Mapped[Optional[str]]
	derivedFrom: Mapped[Optional[str]]


class SoftwareSQL(EntitySQL, HasContent, Versioned, Base):
	__tablename__ = 'software'


class ComputationSQL(EntitySQL, Base):
	__tablename__ = 'computation'

	usedSoftware: Mapped[str]

In [15]:
EntityTypesROCrate = Union[
	ROCrateMetadataElemSQL,
	DatasetSQL,
	SoftwareSQL,
	ComputationSQL,
]

### Test Creating Classes

In [16]:
testGUID = "ark:59853/test"
testChildGUID= "ark:59853/test-member"

KeywordSQL(guid=testGUID, keywordValue='data')

# decomposed rows to create
keywords = [
		KeywordSQL(guid=testGUID, keywordValue='data'),
		KeywordSQL(guid=testGUID, keywordValue='test')
	],
authors= [
		AuthorSQL( name="Max Levinson", orcid="https://orcid.org/0000-0003-0384-8499")
	],

hasPart = [
		MembershipSQL(
			parentGUID=testGUID, 
			parentType=MetadataTypeEnumSQL.ROCRATE,
			childGUID=testChildGUID, 
			childType=MetadataTypeEnumSQL.DATASET
			)	
	]
# Create a test ROCrate with Keywords 
testElem = ROCrateMetadataElemSQL(
	guid= testGUID,
	name="Test ROCrate",
	description="A template ROCrate",
	

)

## Converting `fairscape_models` into SQLAlchemy ORM Models

1. Load an Example ROCrate
2. Need to Create a Class to Contain all Records to create
 - Want to simply return the objects to create

In [17]:

# setup engine for sqlalchemy

# create an engine
engine = sa.create_engine("sqlite:///test.db")

# create table 
Base.metadata.create_all(engine)

In [18]:
dataversePath = pathlib.Path("/mnt/data/Dataverse")
releasePath = dataversePath / 'October2025'

### Load an Example ROCrate

In [19]:

def findCrateRootMetadata(
	cratePath: pathlib.Path,
	)->bytes | None:
	""" Returns bytes json of a zipped ROCrate ro-crate-metadata.json file
	"""

	with zipfile.ZipFile(str(cratePath), 'r') as zip_ref:
		namelist = zip_ref.namelist()

		if 'ro-crate-metadata.json' in namelist:
			return zip_ref.read('ro-crate-metadata.json')

		# TODO issue with subfolders, elements can be inside subfolders without folder existing in namelist
		# i.e. root/ro-crate-metadata.json can exist without root/ appearing in the namelist

		#find root subfolder
		#
		# subfolders = [ elem for elem in namelist if elem.endswith("/") and elem.count("/") == 1]
		# return namelist

		#	TODO exception for multiple root crates in 
		# if len(subfolders) != 1:
		#	raise Exception
		#crateZipPath = subfolders[0] + "ro-crate-metadata.json"
		# TODO exception
		#if crateZipPath not in namelist:
		#	raise Exception

		matchingROCrates = [ elem for elem in namelist if 'ro-crate-metadata.json' in elem]

		if len(matchingROCrates) > 1:
			raise Exception
		else:
			# read the content	
			return zip_ref.read(matchingROCrates[0])


def readCrate(
	cratePath: pathlib.Path,
	) -> ROCrateV1_2 | None:
	""" Given a path to a zipped crate read the rocrate json into an ROCrateV1_2"""
	rootCrateJSON = findCrateRootMetadata(cratePath)
	#rootCrateDict = json.loads(rootCrateJSON)

	try:
		return ROCrateV1_2.model_validate_json(rootCrateJSON)
	except Exception as e:
		return e.errors()

In [20]:
# pick a sample crate to load
dataversePath = pathlib.Path("/mnt/data/Dataverse")
releasePath = dataversePath / 'October2025'

# grab a single rocrate
imageCratePath = releasePath / 'cm4ai_ifimages_MDA-MB-468_paclitaxel_october_2025.zip'

# U2OS crates
cratePath = dataversePath / 'U2OS' / 'cm4ai_u2os_1_ImageDownloader.zip'


In [21]:
inputCrate = readCrate(cratePath)
testCrateElem = inputCrate.metadataGraph[1]

In [22]:
len(inputCrate.metadataGraph)

20552

In [23]:
testDatasetElem = inputCrate.getDatasets()[0]
testSoftwareElem = inputCrate.getSoftware()[0]
testComputationElem = inputCrate.getComputations()[0]

### Dumping Pydantic Models to SQL Alchemy Classes

In [24]:

# TODO Simplify Dumping using **kwargs approach

entityKeys = [
	"guid",
	"name",
	"description",
]


roCrateKeys = entityKeys + ["version"]
datasetKeys = entityKeys + ["version", "fileFormat", "datePublished"]
softwareKeys = entityKeys + ["version"]
computationKeys = entityKeys + []

# rocrate construct
stripModel = lambda inputData, keyList: { key: inputData.__dict__[key] for key in keyList}


In [25]:
# rocrate elem
constructedROCrateElem = ROCrateMetadataElemSQL(**stripModel(testCrateElem, roCrateKeys))

# software construct
outputSoftwareSQL = SoftwareSQL(**stripModel(testSoftwareElem, softwareKeys))

# dataset construct
outputDatasetSQL = DatasetSQL(**stripModel(testDatasetElem, datasetKeys))

# computation construct
outputComputationSQL = ComputationSQL(**stripModel(testComputationElem, computationKeys))

In [26]:
# named tupl

In [27]:
import re

In [95]:
def extractAuthor(author):
	if isinstance(author, str):
		# TODO deal with string of list of authors
		return (author, None)
	if isinstance(author, IdentifierValue):
		return (author.name, author.guid)	

def transformAuthors(inputModel):
	""" Convert fairscape_models pydantic element authors into list of SQL authors
	"""
	authorOutput = set()

	if isinstance(inputModel, Computation):
		authorOutput.add(extractAuthor(inputModel.runBy))

	else:
		if isinstance(inputModel.author, list):
			for auth in inputModel.author:
				authorOutput.add(extractAuthor(auth))
		else:
			authorList = re.split(r'[,&]', inputModel.author)
			if len(authorList) == 1:
				authorOutput.add(extractAuthor(inputModel.author))
			else:
				for auth in authorList:
					authorOutput.add((auth.lstrip(" "), None) )

	return authorOutput	

In [29]:
authorList = re.split(r'[,&]', testCrateElem.author)

In [105]:
# yeild authors for all guids
def getInsertDataAuthor(inputCrate):
	authorData = set()
	for elem in inputCrate.metadataGraph:
		elemSQLType = determineMetadataType(elem.metadataType)
		if isinstance(elem, ROCrateMetadataFileElem):
			continue
		match elemSQLType:
			case MetadataTypeEnumSQL.COMPUTATION:
				authorData.add((elem.runBy, None))
			case MetadataTypeEnumSQL.SCHEMA:
				pass
			case _:
				for outputElem in transformAuthors(elem):
					authorData.add(outputElem)
	return authorData

In [33]:
authorData

{('Abantika Pal', None),
 ('Aji Palar', None),
 ('Andrej Sali', None),
 ('Andrew P. Latham', None),
 ('Anthony Cesnik', None),
 ('Christopher Churas', None),
 ('Dexter Pratt', None),
 ('Dorothy Tsai', None),
 ('Edward L. Huttlin', None),
 ('Emma Lundberg ', None),
 ('Ernst Pulido', None),
 ('Gege Qian', None),
 ('Ignacia Echeverria', None),
 ('Ishan Gaur', None),
 ('J. Wade Harper', None),
 ('Jing Chen', None),
 ('Joanna Lenkiewicz', None),
 ('Katherine Licon', None),
 ('Keiichiro Ono', None),
 ('Kyung-Mee Moon', None),
 ('Laura Pontano Vaites', None),
 ('Leah V. Schaffer', None),
 ('Leonard J. Foster', None),
 ('Mengzhou Hu', None),
 ('Neelesh Soni', None),
 ('Nicole M. Mattson', None),
 ('Peter Zage', None),
 ('Robin Bachelder', None),
 ('Steven P. Gygi', None),
 ('Trang Le', None),
 ('Trey Ideker', None),
 ('William Leineweber', None),
 ('Xiaoyu Zhao', None),
 ('Yue Qin', None)}

In [70]:
# format authorData to go into tables

# data for bulk execute
# authorInsertData = [{"name": auth[0], "orcid": auth[1] } for auth in authorData]

# bulk insert for speed
#with sa.orm.Session(engine) as session:
#	session.execute(sa.insert(AuthorSQL), authorInsertData)
#	session.commit()

# add all data

with sa.orm.Session(engine) as session:
	authorInsertData = [AuthorSQL(**{"name": auth[0], "orcid": auth[1] }) for auth in authorData]
	session.add_all(authorInsertData)
	session.flush()
	author_ids = set([ (item.id, item.name) for item in authorInsertData])
	session.commit()


In [58]:
with sa.orm.Session(engine) as session:
	results = session.scalars(sa.select(AuthorSQL)).all()

In [69]:
# clear the database of records
session = sa.orm.Session(engine)

results = session.scalars(sa.select(AuthorSQL)).all()
for author in results:
	session.delete(author)
session.commit()
session.close()

#### Author -> Identifier Links for each identifier

In [ ]:
identifiers = set()
identifier_authors = set()


for elem in inputCrate.metadataGraph[0:10]:
	if isinstance(elem, ROCrateMetadataFileElem):
		continue
	
	elemSQLType = determineMetadataType(elem.metadataType)
	identifiers.add((elem.guid, elemSQLType, elem.name))

def getLinkedAuthors(inputElem):
	author_ids = [ crate_authors[auth[0]] for auth in transformAuthors(inputElem)]
	author_identifier_insert_data = [{
		"author_id": auth_id, 
		"identifier_guid": inputElem.guid
		} for auth_id in author_ids]
	return author_identifier_insert_data

# link all elements to the authors
def generateLinkedAuthors(inputCrate):
	for element in inputCrate.metadataGraph:
		if isinstance(element, ROCrateMetadataFileElem):
			continue
		if isinstance(element, Schema):
			continue
		else:
			yield getLinkedAuthors(element)

AttributeError: 'Computation' object has no attribute 'author'

In [88]:
crate_authors = {auth[1]: auth[0] for auth in author_ids}

In [92]:
getAuthors(inputCrate.metadataGraph[10])

[{'author_id': 1, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 2, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 3, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 4, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 5, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 6, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 7, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 8, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 9, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 10, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 11, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 12, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 13, 'identifier_guid': 'ark:59853/dataset-image-1193_d6_5'},
 {'author_id': 14, 'i

In [104]:
with sa.orm.Session(engine) as session:
	for linked_authors in generateLinkedAuthors(inputCrate):
		session.execute(sa.insert(AuthorIdentifierSQL), linked_authors)
	
	session.flush()
	session.commit()


#### Iterate over all ROCrate elements

In [31]:
def determineMetadataType(inputType: str | List[str])-> MetadataTypeEnumSQL:
	if isinstance(inputType, str):
		inputType = [inputType]

	# TODO rewrite
	if any(['ROCrate' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.ROCRATE
	if any(['Dataset' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.DATASET
	if any(['Software' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.SOFTWARE
	if any(['Computation' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.COMPUTATION
	if any(['Schema' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.SCHEMA
	if any(['BioChemEntity' in elem for elem in inputType]):
		return MetadataTypeEnumSQL.BIO_CHEM_ENTITY

In [16]:

def convertROCrateElements(inputCrate: ROCrateV1_2)->list:
	metadataElements = []
	identifierTableElements = []
	keywordElements = []
	authorElements = []


	for elem in inputCrate.metadataGraph:
		elemSQLType = determineMetadataType(elem.metadataType)

		if isinstance(elem, ROCrateMetadataFileElem):
			continue

		if isinstance(elem, BioChemEntity):
			continue


		match elemSQLType:

			# TODO: Low Priority Types
			case MetadataTypeEnumSQL.MEDICAL_CONDITION:
				pass
			case MetadataTypeEnumSQL.ORGANIZATION:
				pass
			case MetadataTypeEnumSQL.ANNOTATION:
				pass
			case MetadataTypeEnumSQL.CREATIVE_WORK:
				pass

			# TODO: High Prio
			case MetadataTypeEnumSQL.SAMPLE:
				pass
			case MetadataTypeEnumSQL.BIO_CHEM_ENTITY:
				pass
			case MetadataTypeEnumSQL.EXPERIMENT:
				pass
			case MetadataTypeEnumSQL.SCHEMA:
				pass

			case MetadataTypeEnumSQL.ROCRATE:
				identifierTableElements.append(IdentifiersSQL(
					guid=elem.guid,
					name=elem.name,
					metadataType=MetadataTypeEnumSQL.ROCRATE
				))
				metadataElements.append(ROCrateMetadataElemSQL(**stripModel(elem, roCrateKeys)))
				keywordElements += [ KeywordSQL(guid=elem.guid, keywordValue=keywordElem) for keywordElem in elem.keywords]
				authorElements += transformAuthors(elem)

			case MetadataTypeEnumSQL.DATASET:
				identifierTableElements.append(IdentifiersSQL(
					guid=elem.guid,
					name=elem.name,
					metadataType=MetadataTypeEnumSQL.DATASET
				))
				metadataElements.append(DatasetSQL(**stripModel(elem, datasetKeys)))
				keywordElements += [ KeywordSQL(guid=elem.guid, keywordValue=keywordElem) for keywordElem in elem.keywords]
				authorElements += transformAuthors(elem)
			case MetadataTypeEnumSQL.SOFTWARE:
				identifierTableElements.append(IdentifiersSQL(
					guid=elem.guid,
					name=elem.name,
					metadataType=MetadataTypeEnumSQL.SOFTWARE
				))
				metadataElements.append(SoftwareSQL(**stripModel(elem, softwareKeys)))
				authorElements += transformAuthors(elem)
			case MetadataTypeEnumSQL.COMPUTATION:
				identifierTableElements.append(IdentifiersSQL(
					guid=elem.guid,
					name=elem.name,
					metadataType=MetadataTypeEnumSQL.COMPUTATION
				))
				metadataElements.append(ComputationSQL(**stripModel(elem, computationKeys)))
				authorElements += [AuthorSQL(guid=elem.guid, name=elem.runBy)]
	
	return metadataElements + identifierTableElements + keywordElements + authorElements

In [ ]:
with sa.orm.Session(engine) as session:
	session.execute(constructedROCrateElem)
	session.commit()

In [14]:
testCrateElem.keywords

['AI',
 'artificial intelligence',
 'breast cancer',
 'cell maps',
 'CM4AI',
 'machine learning',
 'MDA-MB-468',
 'paclitaxel',
 'protein localization',
 'protein localization']

### Converting Entity properties into AuthorsSQL

In [ ]:
def transformAuthors(inputModel):
	""" Convert fairscape_models pydantic element authors into list of SQL authors
	"""
	authorOutput = []
	for auth in inputModel.author:
		if isinstance(auth, str):
			authorOutput.append(
				AuthorSQL(
					guid=inputModel.guid, 
					name=auth
				)	
			)
		if isinstance(auth, IdentifierValue):
			authorOutput.append(
				AuthorSQL(
					guid=inputModel.guid, 
					name=auth.name, 
					orcid=auth.guid
				)	
			)

	return authorOutput	



def convertROCrateMetadataToSQL(
	inputCrateElem: ROCrateMetadataElem
	)-> List[ROCrateMetadataElemSQL | AuthorSQL | KeywordSQL]:
	""" Convert Basic ROCrateMetadata Elem to ROCrateMetadataElemSQL 
	TODO: Doesn't Handle hasPart relation, requires to look at the whole ROCrate
	"""

	authorOutput = []
	for auth in inputCrateElem.author:
		if isinstance(auth, str):
			authorOutput.append(
				AuthorSQL(
					guid=inputCrateElem.guid, 
					name=auth
				)	
			)
		if isinstance(auth, IdentifierValue):
			authorOutput.append(
				AuthorSQL(
					guid=inputCrateElem.guid, 
					name=auth.name, 
					orcid=auth.guid
				)	
			)
	
	keywords = [
		KeywordSQL(
			guid = inputCrateElem.guid, 
			keywordValue = keywordValue
			)
		for keywordValue in inputCrateElem.keywords
	]

	return [ ROCrateMetadataElemSQL(
		guid= inputCrateElem.guid,
		name= inputCrateElem.name,
		description= inputCrateElem.description,
		version= inputCrateElem.version,
		license= inputCrateElem.dataLicense,
		# TODO no contentURL on ROCrate
		contentURL= inputCrateElem.url,
	)] + keywords + authorOutput


def getRootCrateMembership(inputCrate: ROCrateV1_2, rootCrateGUID: str) -> List[MembershipSQL]:
	""" Returns membership values for root ROCrateMetadataElemSQL.hasPart
	"""
	membershipRecords = []
	for metadataElem in inputCrate.metadataGraph:
		if isinstance(metadataElem, ROCrateMetadataFileElem):
			pass
		elif isinstance(metadataElem, ROCrateMetadataElem):
		# TODO: handle nested rocrates
			pass
		else:
			# create hasPart references
			# child type 
			membershipRecords.append(MembershipSQL(
				parentGUID= rootCrateGUID, 
				parentType=MetadataTypeEnumSQL.ROCRATE,
				childGUID=metadataElem.guid, 
				childType= determineMetadataType(metadataElem.metadataType)
				)	
			)
	return membershipRecords

In [16]:
# TODO find root crate function
def findRootCrate(inputCrate: ROCrateV1_2) -> ROCrateMetadataElem:
	pass

In [17]:



def convertSoftwareToORM(
	inputSoftware: Software
	)->SoftwareSQL:
	return SoftwareSQL(
		guid=inputSoftware.guid,
		name=inputSoftware.name,
		version=inputSoftware.version,
		#license=inputSoftware.dataLicense,
		contentURL=inputSoftware.contentUrl
	)

def convertDatasetToORM(inputDataset: Dataset)->DatasetSQL:
	return DatasetSQL(
		guid=inputDataset.guid,
		name=inputDataset.name,
		version=inputDataset.version,
		#license=inputDataset.dataLicense,
		contentURL=inputDataset.contentUrl
	)

def convertComputationToORM(inputComputation: Computation)->ComputationSQL:
	return ComputationSQL(
		guid=inputComputation.guid,
		name=inputComputation.name,
		version=inputComputation.version,
		#license=inputComputation.dataLicense,
		contentURL=inputComputation.contentUrl
	)

## Create ROCrate Elements

### Writing Input ROCrate to SQL

In [19]:

entities = convertROCrateMetadataToSQL(testCrateElem)
rootCrateGUID = testCrateElem.guid
membershipEntities = getRootCrateMembership(inputCrate, rootCrateGUID)

# create an identifier record
identifierRecord = IdentifiersSQL(
	guid=rootCrateGUID,
	name=testCrateElem.guid,
	metadataType=MetadataTypeEnumSQL.ROCRATE
)

metadataGraphElements = convertROCrateElements(inputCrate)

records = [identifierRecord] + entities + membershipEntities + metadataGraphElements




In [20]:

# create the rocrate instance
with sa.orm.Session(engine) as session:
	session.add_all(records)
	session.commit()

In [ ]:
# clear the database of records
with sa.orm.Session(engine) as session:
	for rec in records:
		try:
			session.delete(rec)
		except:
			pass
	session.commit()

In [ ]:
# clear the identifier table
with sa.orm.Session(engine) as session:
	query = sa.select(IdentifiersSQL)
	for elem in session.scalars(query).all():
		session.delete(elem)
	
	session.commit()

In [ ]:
convertedMetadataElem.keywords

In [ ]:
testGUID

'ark:59853/test'

In [21]:
with sa.orm.Session(engine) as session:
	query = sa.select(AuthorSQL).filter_by(guid=rootCrateGUID)
	for elem in session.scalars(query).all():
		print(elem.name)


Hansen JN
Axelsson U
Johannesson A
Fall J
Ballllosera Navarro F
Lundberg E
Hansen JN
Axelsson U
Johannesson A
Fall J
Ballllosera Navarro F
Lundberg E


In [ ]:
with sa.orm.Session(engine) as session:
	query = sa.select(KeywordSQL)
	for elem in session.scalars(query):
		print(f"{elem.guid} {elem.keywordValue}")

ark:59853/rocrate-paclitaxel-if-data-release AI
ark:59853/rocrate-paclitaxel-if-data-release artificial intelligence
ark:59853/rocrate-paclitaxel-if-data-release breast cancer
ark:59853/rocrate-paclitaxel-if-data-release cell maps
ark:59853/rocrate-paclitaxel-if-data-release CM4AI
ark:59853/rocrate-paclitaxel-if-data-release machine learning
ark:59853/rocrate-paclitaxel-if-data-release MDA-MB-468
ark:59853/rocrate-paclitaxel-if-data-release paclitaxel
ark:59853/rocrate-paclitaxel-if-data-release protein localization
ark:59853/rocrate-paclitaxel-if-data-release protein localization


In [ ]:
convertedMetadataElem.guid

'ark:59853/rocrate-paclitaxel-if-data-release'

In [ ]:
ROCrateMetadataElemSQL

### Return an ROCrate Element

In [57]:
TYPE_LOOKUP = {
	MetadataTypeEnumSQL.DATASET: DatasetSQL,
	MetadataTypeEnumSQL.SOFTWARE: SoftwareSQL,
	MetadataTypeEnumSQL.ROCRATE: ROCrateMetadataElemSQL,
	MetadataTypeEnumSQL.COMPUTATION: ComputationSQL
}

In [ ]:
class QueryByGUID():
	def __init__(self, guid: str):
		self.guid = guid

	def _query_type(self, session: sa.orm.Session) -> MetadataTypeEnumSQL:

		# query the identifier table to get the metadata type
		q = sa.select(IdentifiersSQL).filter_by(guid=self.guid)
		identifierResults = session.scalars(q).all()

		return identifierResults[0].metadataType

	def execute(self, session: sa.orm.Session):

		# lookup the root query
		rootEntityType = TYPE_LOOKUP[self._query_type(session)]
		rootEntityQuery = sa.select(rootEntityType).filter_by(guid=self.guid)

		rootEntityResults = session.scalars(rootEntityQuery).all()
	
		# get keywords
		queryKeywords = sa.select(KeywordSQL).filter_by(guid=self.guid)
		keywordResults = session.scalars(queryKeywords).all()

	# TODO get authors
		queryAuthors = sa.select(AuthorSQL).filter_by(guid=self.guid)
		authorResults = session.scalars(queryAuthors).all()

		# get hasPart
		queryHasPart = sa.select(MembershipSQL).filter_by(parentGUID=self.guid)
		hasPartResults = session.scalars(queryHasPart).all()

		# isPartOf
		queryIsPartOf = sa.select(MembershipSQL).filter_by(childGUID=self.guid)
		isPartOfResults = session.scalars(queryIsPartOf).all()

		# TODO Provenance properties

		return QueryResponse(
			rootEntityResults, 
			keywordResults, 
			authorResults, 
			hasPartResults, 
			isPartOfResults
		)


In [ ]:
class QueryResponse():

	def __init__(
		self, 
		rootEntity: Union[ROCrateMetadataElem, SoftwareSQL, DatasetSQL, ComputationSQL],
		authorResults: List[AuthorSQL],
		keywordResults: List[KeywordSQL],
		hasPartResults: List[MembershipSQL],
		isPartOfResults: List[MembershipSQL]
		):

		self.rootEntity = rootEntity
		self.authorResults = authorResults
		self.keywordResults = keywordResults
		self.hasPartResults = hasPartResults
		self.isPartOfResults = isPartOfResults
		self.metadata = None


	def _transform_authors(self):
		authorsList = []
		for elem in self.authorResults:
			if elem.orcid:
				authorsList.append({"@id": elem.orcid})
			else:
				authorsList.append(elem.name)
		return authorsList

	def _transform_root_entity(self):	
		metadata = self.rootEntityResults[0].__dict__.copy()
		del metadata['_sa_instance_state']
		del metadata['id']
		metadata['@id'] = metadata.pop("guid")
		metadata['@type'] = ["EVI:Dataset", "https://schema.org/Dataset"]
		self.metadata = metadata

	def transform(self):

		self._transform_root_entity()
		authorList = self._transform_authors()

		self.metadata = {
			**self.metadata,
			"author": authorList,
			"keywords": [ elem.keywordValue for elem in self.keywordResults],
			"hasPart": [ {"@id": elem.childGUID} for elem in self.hasPartResults],
			"isPartOf": [{"@id": elem.parentGUID} for elem in self.isPartOfResults]
		}


class ROCrateQueryResponse(QueryResponse):
	def __init__(
		self, 
		crateResults, 
		keywordResults, 
		authorResults, 
		hasPartResults,
		isPartOfResults
	):
		self.crateResults = crateResults
		self.keywordResults = keywordResults
		self.authorResults = authorResults
		self.hasPartResults = hasPartResults
		self.isPartOfResults = isPartOfResults
		self.metadata = None

	def transform(self):
		# use crate results dict to form the metadata
		metadata = self.crateResults[0].__dict__.copy()
		del metadata['_sa_instance_state']
		del metadata['id']
		metadata['@id'] = metadata.pop("guid")
		metadata['@type'] = ["EVI:Dataset", "https://schema.org/Dataset"]
		
		authorList = self._transform_authors()

		self.metadata = {
			**metadata,
			"author": authorList,
			"keywords": [ elem.keywordValue for elem in self.keywordResults],
			"hasPart": [ {"@id": elem.childGUID} for elem in self.hasPartResults],
			"isPartOf": [{"@id": elem.parentGUID} for elem in self.isPartOfResults]
		}


	

In [23]:
def queryROCrate(
	queryGUID: str, 
	session: sa.orm.Session
	)->ROCrateQueryResponse:
	""" Given a GUID query rocrate and all properties
	"""
	queryROCrate = sa.select(ROCrateMetadataElemSQL).filter_by(guid=queryGUID)
	crateResults = session.scalars(queryROCrate).all()
	
	# get keywords
	queryKeywords = sa.select(KeywordSQL).filter_by(guid=queryGUID)
	keywordResults = session.scalars(queryKeywords).all()

# get authors
	queryAuthors = sa.select(AuthorSQL).filter_by(guid=queryGUID)
	authorResults = session.scalars(queryAuthors).all()

	# get hasPart
	queryHasPart = sa.select(MembershipSQL).filter_by(parentGUID=queryGUID)
	hasPartResults = session.scalars(queryHasPart).all()

	# isPartOf
	queryIsPartOf = sa.select(MembershipSQL).filter_by(childGUID=queryGUID)
	isPartOfResults = session.scalars(queryIsPartOf).all()

	# TODO Provenance properties

	return ROCrateQueryResponse(crateResults, keywordResults, authorResults, hasPartResults, isPartOfResults)



In [24]:
with sa.orm.Session(engine) as session:
	response = queryROCrate(session=session, queryGUID=rootCrateGUID)

In [36]:
response

In [39]:
response.crateResults

[]

In [25]:
response.transform()

In [26]:
outputROCrate = ROCrateMetadataElem.model_validate(response.metadata)

### TODO Handling Multiple ROCrates 

## Return Software 

In [27]:
with sa.orm.Session(engine) as session:
	querySoftware = sa.select(SoftwareSQL)
	foundSoftware = session.scalars(querySoftware).all()

In [28]:
foundSoftware

[]

In [24]:
querySoftware

In [29]:
def executeQuery(
	query: sa.sql.selectable.Select, 
	session: sa.orm.Session
):
	return session.scalars(query).all()

In [54]:
querySoftware = sa.select(SoftwareSQL)
queryComputation = sa.select(ComputationSQL)
queryDataset = sa.select(DatasetSQL)
queryAuthors = sa.select(AuthorSQL)
queryKeywords = sa.select(KeywordSQL)
queryIsPartOf = sa.select(MembershipSQL)

def queryKeywordsByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):
	return executeQuery(
		queryKeywords.filter_by(guid=queryGUID),
		session
	)

def queryAuthorsByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):
	return executeQuery(
		queryAuthors.filter_by(guid=queryGUID),
		session
	)

def queryIsPartOfByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):
	return executeQuery(
		queryIsPartOf.filter_by(childGUID=queryGUID),
		session
	)

def queryHasPartByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):
	return executeQuery(
		queryIsPartOf.filter_by(parentGUID=queryGUID),
		session
	)

def querySoftwareByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):

	query = querySoftware.filter_by(guid=queryGUID)
	return executeQuery(query, session)

def queryDatasetByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):

	query = queryDataset.filter_by(guid=queryGUID)
	return executeQuery(query, session)

def queryComputationByGUID(
	queryGUID: str, 
	session: sa.orm.Session
):

	query = queryComputation.filter_by(guid=queryGUID)
	return executeQuery(query, session)


In [31]:
with sa.orm.Session(engine) as ses:
	q = queryDataset
	datasets = executeQuery(q, ses)

In [32]:
datasetElem = datasets[0]

In [33]:
def cleanElem(input):
	""" Converting a single node into Dictionary
	"""

	metadata = input.__dict__.copy()
	del metadata['_sa_instance_state']
	del metadata['id']
	metadata['@id'] = metadata.pop("guid")

	return metadata

In [37]:

metadata = cleanElem(datasetElem)
metadata['@type'] = "EVI:Dataset"

In [ ]:
def queryByGUID(
	metadataType: MetadataTypeEnumSQL, 
	queryGUID: str, 
	ses: sa.orm.Session
	)-> QueryResponse:
	""" Query all metadata for a single node by GUID
	"""

	match metadataType:
		case MetadataTypeEnumSQL.ROCRATE:
			query = sa.select(ROCrateMetadataElem).filter_by(guid=queryGUID)

		#case MetadataTypeEnumSQL.SCHEMA:
			# TODO support SCHEMA
			# query = sa.select(ROCrateMetadataElem).filter_by(guid=queryGUID)
			pass
		case MetadataTypeEnumSQL.COMPUTATION:
			query = sa.select(ComputationSQL).filter_by(guid=queryGUID)

		case MetadataTypeEnumSQL.DATASET:
			query = sa.select(DatasetSQL).filter_by(guid=queryGUID)

		case MetadataTypeEnumSQL.SOFTWARE:
			query = sa.select(SoftwareSQL).filter_by(guid=queryGUID)

	rootEntityResults = executeQuery(query,session=ses)
	
	authorResults = queryAuthorsByGUID(
		queryGUID=datasetElem.guid,
		session=ses
		)

	keywordResults = queryKeywordsByGUID(
		queryGUID=datasetElem.guid,
		session=ses
	)	

	hasPartResults = queryHasPartByGUID(	
		queryGUID=datasetElem.guid,
		session=ses
	)

	isPartOfResults = queryIsPartOfByGUID(	
		queryGUID=datasetElem.guid,
		session=ses
	)
	
	# TODO provenance for 

	return QueryResponse(
		rootEntity=rootEntityResults,
		authorResults=authorResults,
		keywordResults=keywordResults,
		hasPartResults=hasPartResults,
		isPartOfResults=isPartOfResults
	)

In [47]:
# get keywords and authors
with sa.orm.Session(engine) as ses:
	authorResults = queryAuthorsByGUID(
		queryGUID=datasetElem.guid,
		session=ses
		)

	keywordResults = queryKeywordsByGUID(
		queryGUID=datasetElem.guid,
		session=ses
	)	

	isPartOfResults = queryIsPartOfByGUID(	
		queryGUID=datasetElem.guid,
		session=ses
	)

In [48]:
authorsList = []
for elem in authorResults:
	if elem.orcid:
		authorsList.append({"@id": elem.orcid})
	else:
		authorsList.append(elem.name)

In [49]:
dataset_output ={ **metadata,
	"author": authorsList,
	"keywords": [ elem.keywordValue for elem in keywordResults],
	#"hasPart": [ {"@id": elem.childGUID} for elem in hasPartResults],
	"isPartOf": [{"@id": elem.parentGUID} for elem in isPartOfResults]
}

In [53]:
dataset_output

{'derivedFrom': None,
 'name': 'Paclitaxel Manifest',
 'version': '0.1.0',
 'generatedBy': None,
 'description': None,
 'contentURL': 'file:///manifest.csv',
 '@id': 'ark:59853/dataset-paclitaxel-manifest',
 '@type': 'EVI:Dataset',
 'author': ['Hansen JN'],
 'keywords': ['if images', 'metadata'],
 'isPartOf': [{'@id': 'ark:59853/rocrate-paclitaxel-if-data-release'}]}

In [51]:
Dataset.model_validate(dataset_output)

ValidationError: 3 validation errors for Dataset
description
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
datePublished
  Field required [type=missing, input_value={'derivedFrom': None, 'na...axel-if-data-release'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
format
  Field required [type=missing, input_value={'derivedFrom': None, 'na...axel-if-data-release'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [ ]:
def convertSoftware():
	pass

## ETC.

In [39]:
with sa.orm.Session(engine) as session:
	query = sa.select(ROCrateMetadataElemSQL).filter_by(guid=rootCrateGUID)
	results = session.scalars(query).all()

In [42]:
for elem in results:
	print(elem.guid)

ark:59853/rocrate-paclitaxel-if-data-release


In [52]:
results[0].description
results[0].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x709a41ee7f20>,
 'id': 1,
 'guid': 'ark:59853/rocrate-paclitaxel-if-data-release',
 'name': 'Paclitaxel IF Images',
 'description': 'This data set displays the spatial localization of 464 proteins of interest in cells of the breast cancer cell line MDA-MB-468 treated with paclitaxel as imaged by immunofluorescence-based staining (ICC-IF) and confocal microscopy in the Lundberg Lab at Stanford University, as part of the Cell Maps for Artificial Intelligence (CM4AI; CM4AI.org) project. Nuclei were stained with DAPI (blue channel); endoplasmic reticulum with a calreticulin antibody (yellow channel); microtubules with tubulin antibody (red channel); and antibody against protein of interest (green channel). \n\nThis data is Copyright (c) 2025 The Board of Trustees of the Leland Stanford Junior University. It is licensed for reuse under Creative Commons Attribution ShareAlike NonCommercial 4.0 International License (https://creat

In [62]:
# remove a property of 
metadata = results[0].__dict__.copy()
del metadata['_sa_instance_state']
del metadata['id']

In [63]:
metadata

{'guid': 'ark:59853/rocrate-paclitaxel-if-data-release',
 'name': 'Paclitaxel IF Images',
 'description': 'This data set displays the spatial localization of 464 proteins of interest in cells of the breast cancer cell line MDA-MB-468 treated with paclitaxel as imaged by immunofluorescence-based staining (ICC-IF) and confocal microscopy in the Lundberg Lab at Stanford University, as part of the Cell Maps for Artificial Intelligence (CM4AI; CM4AI.org) project. Nuclei were stained with DAPI (blue channel); endoplasmic reticulum with a calreticulin antibody (yellow channel); microtubules with tubulin antibody (red channel); and antibody against protein of interest (green channel). \n\nThis data is Copyright (c) 2025 The Board of Trustees of the Leland Stanford Junior University. It is licensed for reuse under Creative Commons Attribution ShareAlike NonCommercial 4.0 International License (https://creativecommons.org/licenses/by-nc-sa/4.0/). Attribution is required to the copyright holders 

In [55]:
with sa.orm.Session(engine) as session:
	# get keywords
	queryKeywords = sa.select(KeywordSQL).filter_by(guid=rootCrateGUID)
	keywordResults = session.scalars(queryKeywords).all()

# get authors
	queryAuthors = sa.select(AuthorSQL).filter_by(guid=rootCrateGUID)
	authorResults = session.scalars(queryAuthors).all()

	# get hasPart
	queryHasPart = sa.select(MembershipSQL).filter_by(parentGUID=rootCrateGUID)
	hasPartResults = session.scalars(queryHasPart).all()

	# isPartOf
	queryIsPartOf = sa.select(MembershipSQL).filter_by(childGUID=rootCrateGUID)
	isPartOfResults = session.scalars(queryIsPartOf).all()




In [58]:
authorResults
isPartOfResults
hasPartResults

 ...]

In [70]:
len(hasPartResults)

22576

### Convert an ROCrate Element into `fairscape_models`

In [29]:
metadataElements

In [ ]:
# for every element in the metadata graph


22576

In [ ]:
# ROCrate Elem
crateElem.metadataType

['Dataset', 'https://w3id.org/EVI#ROCrate']

In [ ]:
# Software Elem


In [ ]:
# for every crate elem

In [ ]:
# fairscape_models -> SQLAlchemy
# have method for orm to_orm()

In [ ]:
# SQLAlchemy Models -> fairscape_models 

In [36]:
session = sa.orm.Session(engine)

In [ ]:
session.add_all(metadataElements)
session.commit()

In [39]:
session.commit()

In [45]:
session.close()

In [21]:
query = sa.select(KeywordSQL).where(KeywordSQL.guid.in_([testGUID]))

In [22]:

queryResults = session.execute(query)

In [26]:
result = queryResults.scalars()

[]

In [42]:
query

In [54]:
queryResults.all()

InvalidRequestError: Object <KeywordSQL at 0x7f8966befb70> cannot be converted to 'persistent' state, as this identity map is no longer valid.  Has the owning Session been closed? (Background on this error at: https://sqlalche.me/e/20/lkrp)